In [ ]:
!pip install -U crewai

In [ ]:
!pip install 'crewai[google-genai]'

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 724.4/724.4 kB 15.1 MB/s eta 0:00:00
  Attempting uninstall: google-genai
    Found existing installation: google-genai 1.68.0
    Uninstalling google-genai-1.68.0:
      Successfully uninstalled google-genai-1.68.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.29.0 requires opentelemetry-api<1.39.0,>=1.36.0, but you have opentelemetry-api 1.34.1 which is incompatible.
google-adk 1.29.0 requires opentelemetry-exporter-otlp-proto-http>=1.36.0, but you have opentelemetry-exporter-otlp-proto-http 1.34.1 which is incompatible.
google-adk 1.29.0 requires opentelemetry-sdk<1.39.0,>=1.36.0, but you have opentelemetry-sdk 1.34.1 which is incompatible.
google-adk 1.29.0 requires pydantic<3.0.0,>=2.12.0, but you have pydantic 2.11.10 

In [ ]:
from crewai import Agent,LLM


In [ ]:
from google.colab import userdata

In [ ]:
gemini_api_key=userdata.get("Gemini_api_key")

In [ ]:
llm=LLM (
        "gemini/gemini-2.5-flash",
        api_key=gemini_api_key
)

In [ ]:

game_designer=Agent(  role="Creative Game Designer",

   goal="Come up with fun, feasible game concepts and detailed mechanics based on user idea",
   backstory=
     "You are an experienced game designer."
     "You excel at turning vague ideas into clear, exciting game designs including:"
     "- core loop, rules, win/lose conditions"
     "- basic entities (player, enemies, items)"
     "- controls and feel"
     "Keep it simple enough to implement in pure Python + Pygame in one file.",
    verbose=True,
    llm=llm
  )

In [ ]:
!pip install 'crewai[tools]'

In [ ]:
from crewai_tools import SerperDevTool

In [ ]:
serper_api_key = userdata.get('SERPER_API_KEY')

search_tool = SerperDevTool(api_key=serper_api_key)

In [ ]:
senior_engineer = Agent(
   role="Senior Python Game Developer",
   goal="Write clean, working Python code (using Pygame) for the described game",
    backstory=
        "You are a senior software engineer specialized in Python game development with Pygame."
        "You write structured, readable code with:"
        "- Proper game loop, event handling, drawing"
        "- Comments explaining key parts"
        "- Error handling where needed"
        "You always produce a complete, runnable .py file.",
   verbose=True,
   tools = [search_tool],
   llm=llm,
)

In [ ]:
qa_engineer = Agent(
    role="QA Engineer & Code Reviewer",
    goal="Test, review, and improve the code for bugs, playability, and completeness",
    backstory=
        "You are a meticulous QA engineer and code reviewer."
        "You carefully check:"
        "- Does the code run without errors?"
        "- Does it implement ALL the designed features?"
        "- Is it fun/playable? Any obvious balance issues?"
        "- Code style, variable names, comments"
        "Suggest fixes or small improvements and output the FINAL improved code.",
    verbose=True,
    llm=llm,
)

In [ ]:
from crewai import Task


In [ ]:
task_design = Task(
    description=
        "Take the user's game idea: {game_idea}"
        "1. Clarify and expand it into a fun, simple 2D game"
        "2. Describe: objective, controls, entities, win/lose"
        "3. Keep scope small (one level, basic mechanics)"
        "Output format:"
        "## Game Design Document"
        "- Title: ..."
        "- Genre: ..."
        "- Objective: ..."
        "- Controls: ..."
        "- Entities: ..."
        "- Mechanics: ...",
    expected_output="A clear markdown Game Design Document",
    agent=game_designer
)

In [ ]:
task_code = Task(
    description=
        "Using the game design from the previous task"
        "Write a COMPLETE, standalone Python script using Pygame that implements the game."
        "- Include import pygame, sys, random (if needed)"
        "- Full game loop, init, events, update, draw"
        "- Make it runnable with python game.py"
        "- Add simple comments"
        "- The main game loop must be exposed in the python code, it should not be inside any function like main"
        "- Final answer MUST be ONLY the Python code and Instructions on how to play the game",
    expected_output="A complete runnable Pygame Python script",
    agent=senior_engineer,
    context=[task_design]
)

In [ ]:
task_review = Task(
    description=
        "Review the Python code from the previous task."
        "1. Check for syntax/runtime errors"
        "2. Verify it matches the design document"
        "3. Test mentally: does it have init, loop, quit handling, drawing?"
        "4. Suggest fixes/improvements if needed"
        "5. Output the FINAL, improved, ready-to-run code"
        "Your final answer MUST be ONLY the complete Python code along with the instructions on how to play the game",
    expected_output="Final polished, runnable Pygame Python script and instructions on how to play the game",
    agent=qa_engineer,
    context=[task_design, task_code]
)

In [ ]:
from crewai import Crew, Process

game_crew = Crew(
    agents=[game_designer, senior_engineer, qa_engineer],
    tasks=[task_design, task_code, task_review],
    process=Process.sequential,
    verbose=True
)

In [ ]:
game_idea = "A fun endless runner cat where a character jumps over obstacles"
result = game_crew.kickoff(inputs={"game_idea": game_idea})
print(result)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 740da8f5-e006-4bc5-9009-328171f3d918                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Take the user's game idea: A fun endless runner cat where a character jumps over obstacles1. Clarify     │
│  and expand it into a fun, simple 2D game2. Describe: objective, controls, entities, win/lose3. Keep scope      │
│  small (one level, basic mechanics)Output format:## Game Design Document- Title: ...- Genre: ...- Objective:    │
│  ...- Controls: ...- Entities: ...- Mechanics: ...                                                              │
│  ID: 93132671-0d70-4a00-b0e9-873b44603692                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Creative Game Designer                                                                                  │
│                                                                                                                 │
│  Task: Take the user's game idea: A fun endless runner cat where a character jumps over obstacles1. Clarify     │
│  and expand it into a fun, simple 2D game2. Describe: objective, controls, entities, win/lose3. Keep scope      │
│  small (one level, basic mechanics)Output format:## Game Design Document- Title: ...- Genre: ...- Objective:    │
│  ...- Controls: ...- Entities: ...- Mechanics: ...                                                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Creative Game Designer                                                                                  │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ## Game Design Document                                                                                        │
│                                                                                                                 │
│  -   **Title:** Pixel Paws: The Endless Leap                                                                    │
│  -   **Genre:** Endless Runner, 2D Platformer (minimal)                                                         │
│  -   **Objective:** The player controls "Pixel Paws," an agile cat, as it automatically runs across an endless  │
│  landscape. The primary objective is to achieve the highest possible score by running for as long as possible,  │
│  skillfully jumping over incoming obstacles, and collecting shiny yarn balls.                                   │
│  -   **Controls:**                                                                                              │
│      *   **SPACEBAR / UP Arrow Key:** Jump. Pressing the key makes Pixel Paws leap into the air. Releasing (or  │
│  holding) doesn't affect the jump arc once initiated; gravity takes over. There is no double-jump mechanic.     │
│  -   **Entities:**                                                                                              │
│      *   **Player: Pixel Paws (The Cat)**                                                                       │
│          *   A simple 2D sprite of a running cat.                                                               │
│          *   Animation states: Running (default), Jumping (mid-air pose).                                       │
│          *   Always moves right across the screen automatically. Its horizontal position relative to the        │
│  camera is fixed, while the world scrolls left.                                                                 │
│      *   **Obstacles:**                                                                                         │
│          *   **Cardboard Box:** A low, single-unit height obstacle that requires a basic jump to clear.         │
│  Appears randomly from the right edge of the screen.                                                            │
│          *   **Fishbone Pile:** Another low, single-unit height obstacle, functionally identical to the         │
│  Cardboard Box but with a different visual. Appears randomly from the right edge.                               │
│      *   **Collectibles:**                                                                                      │
│          *   **Yarn Ball:** A small, spinning yarn ball sprite. Appears periodically and randomly on the        │
│  ground level. Pixel Paws gains points when touching a Yarn Ball.                                               │
│      *   **Environment:**                                                                                       │
│          *   **Ground:** A solid, flat surface that Pixel Paws runs on. Scrolls from right to left.             │
│          *   **Background:** A simple, scrolling background layer (e.g., stylized city rooftops or park         │
│  scenery) that moves slower than the ground to create a sense of depth (parallax effect).                       │
│  -   **Mechanics:**                                                                                             │
│      *   **Endless Running:** Pixel Paws moves forward 

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Take the user's game idea: A fun endless runner cat where a character jumps over obstacles1. Clarify     │
│  and expand it into a fun, simple 2D game2. Describe: objective, controls, entities, win/lose3. Keep scope      │
│  small (one level, basic mechanics)Output format:## Game Design Document- Title: ...- Genre: ...- Objective:    │
│  ...- Controls: ...- Entities: ...- Mechanics: ...                                                              │
│  Agent: Creative Game Designer                                                                                  │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Review the Python code from the previous task.1. Check for syntax/runtime errors2. Verify it matches     │
│  the design document3. Test mentally: does it have init, loop, quit handling, drawing?4. Suggest                │
│  fixes/improvements if needed5. Output the FINAL, improved, ready-to-run codeYour final answer MUST be ONLY     │
│  the complete Python code along with the instructions on how to play the game                                   │
│  ID: 0db06838-046f-4cae-8342-e1c59241b296                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: QA Engineer & Code Reviewer                                                                             │
│                                                                                                                 │
│  Task: Review the Python code from the previous task.1. Check for syntax/runtime errors2. Verify it matches     │
│  the design document3. Test mentally: does it have init, loop, quit handling, drawing?4. Suggest                │
│  fixes/improvements if needed5. Output the FINAL, improved, ready-to-run codeYour final answer MUST be ONLY     │
│  the complete Python code along with the instructions on how to play the game                                   │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: QA Engineer & Code Reviewer                                                                             │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Here's a comprehensive review and the improved Python code for "Pixel Paws: The Endless Leap," addressing      │
│  bugs, enhancing playability, and ensuring completeness against the design document.                            │
│                                                                                                                 │
│  ### QA Review Summary:                                                                                         │
│                                                                                                                 │
│  1.  **Syntax/Runtime Errors:**                                                                                 │
│      *   No obvious syntax errors.                                                                              │
│      *   **Identified Bug:** The original score calculation logic was flawed, leading to incorrect              │
│  accumulation of collectible points and overwriting the score with distance points. This has been corrected.    │
│      *   **Identified Bug:** The original `Obstacle` and `YarnBall` sprites initialized their `speed` once and  │
│  did not update it as `game_speed` increased. This meant older sprites would move slower than newly spawned     │
│  ones. This has been corrected by passing the current `game_speed` to their `update` methods.                   │
│                                                                                                                 │
│  2.  **Design Document Verification:**                                                                          │
│      *   **All core mechanics (running, jumping, collision, scoring, game over, obstacle/collectible spawning,  │
│  parallax background) are implemented.**                                                                        │
│      *   **Missing from Design:** True sprite animation for Pixel Paws (running/jumping poses) and the          │
│  "spinning" yarn ball were placeholders.                                                                        │
│          *   **Fix:** The code now includes more distinct placeholder visuals for the player (cat-like shape),  │
│  obstacles (box vs. fishbone pile), and **a simple rotation animation for the yarn ball** to simulate spinning  │
│  without requiring external image assets. This significantly improves playability and adherence to the          │
│  "spinning" design.                                                                                             │
│      *   **Missing from Design:** "Stylized city rooftops or park scenery" for background. The parallax         │
│  background was simple ellipses which caused a visual "jump".                                                   │
│          *   **Fix:** The parallax background rendering has been improved to use a seamless scrolling           │
│  technique by drawing two sets of hills. The visual still uses ellipses but the scrolling is now continuous.    │
│      *   **Scoring:** The scoring system now correctly accumulates distance points and collectible bonus        │
│  points.                                                                                                        │
│                                                                                                                 │
│  3.  **Mental Test (Init, Loop, Quit, Drawing):**      

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Review the Python code from the previous task.1. Check for syntax/runtime errors2. Verify it matches     │
│  the design document3. Test mentally: does it have init, loop, quit handling, drawing?4. Suggest                │
│  fixes/improvements if needed5. Output the FINAL, improved, ready-to-run codeYour final answer MUST be ONLY     │
│  the complete Python code along with the instructions on how to play the game                                   │
│  Agent: QA Engineer & Code Reviewer                                                                             │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Here's a comprehensive review and the improved Python code for "Pixel Paws: The Endless Leap," addressing bugs, enhancing playability, and ensuring completeness against the design document.

### QA Review Summary:

1.  **Syntax/Runtime Errors:**
    *   No obvious syntax errors.
    *   **Identified Bug:** The original score calculation logic was flawed, leading to incorrect accumulation of collectible points and overwriting the score with distance points. This has been corrected.
    *   **Identified Bug:** The original `Obstacle` and `YarnBall` sprites initialized their `speed` once and did not update it as `game_speed` increased. This meant older sprites would move slower than newly spawned ones. This has been corrected by passing the current `game_speed` to their `update` methods.

2.  **Design Document Verification:**
    *   **All core mechanics (running, jumping, collision, scoring, game over, obstacle/collectible spawning, parallax background) are implemented.**
    *   **Missi

In [ ]:
%%writefile main.py
import pygame
import sys
import random

# Initialize Pygame
pygame.init()

# --- Constants ---
SCREEN_WIDTH = 800
SCREEN_HEIGHT = 400
FPS = 60

# Colors
BLACK = (0, 0, 0)
WHITE = (255, 255, 255)
GREEN = (0, 255, 0)
RED = (255, 0, 0)
BLUE = (0, 0, 255)
LIGHT_BLUE = (173, 216, 230) # For background sky
BROWN = (139, 69, 19) # For ground
MOUNTAIN_COLOR = (100, 70, 50) # For parallax mountains
CLOUD_COLOR = (200, 200, 200, 180) # For parallax clouds, with some transparency

# Game Physics
GRAVITY = 0.5
PLAYER_JUMP_STRENGTH = -12

# Player properties
PLAYER_WIDTH = 30
PLAYER_HEIGHT = 50
PLAYER_START_X = 50
GROUND_HEIGHT = 50 # Height of the ground rectangle
PLAYER_GROUND_Y = SCREEN_HEIGHT - GROUND_HEIGHT - PLAYER_HEIGHT # Player's Y position when on ground

# Obstacle properties
OBSTACLE_MIN_WIDTH = 20
OBSTACLE_MAX_WIDTH = 40
OBSTACLE_MIN_HEIGHT = 30
OBSTACLE_MAX_HEIGHT = 70 # For ground spikes, will be capped by half below
FLYING_ENEMY_HEIGHT = 25
FLYING_ENEMY_WIDTH = 40
FLYING_ENEMY_Y = SCREEN_HEIGHT - GROUND_HEIGHT - 100 # Mid-air height
MIN_GAP_BETWEEN_OBSTACLES = 150 # Minimum horizontal space between new obstacles

# Game Speed
INITIAL_GAME_SPEED = 5
GAME_SPEED_INCREMENT_PER_SCORE = 0.005 # How much speed increases per score point
MAX_GAME_SPEED = 15

# Obstacle Spawning
INITIAL_OBSTACLE_SPAWN_MIN_INTERVAL = 1000 # milliseconds
INITIAL_OBSTACLE_SPAWN_MAX_INTERVAL = 2500 # milliseconds
SPAWN_INTERVAL_DECREMENT_PER_SCORE = 2 # milliseconds reduction per score point
MIN_SPAWN_INTERVAL_CAP = 300 # Absolute minimum spawn interval to avoid constant spawns

# Background/Parallax properties
BACKGROUND_CLOUD_Y = 50 # Y position for clouds
BACKGROUND_MOUNTAIN_Y = SCREEN_HEIGHT - GROUND_HEIGHT - 80 # Y position for mountains
PARALLAX_SPEED_FACTOR_MOUNTAINS = 0.3
PARALLAX_SPEED_FACTOR_CLOUDS = 0.1

# --- Game Entities ---

class Leapster(pygame.sprite.Sprite):
    def __init__(self):
        super().__init__()
        self.image = pygame.Surface([PLAYER_WIDTH, PLAYER_HEIGHT])
        self.image.fill(BLUE) # Player character color
        self.rect = self.image.get_rect()
        self.rect.x = PLAYER_START_X
        self.rect.y = PLAYER_GROUND_Y
        self.y_velocity = 0
        self.on_ground = True

    def jump(self):
        if self.on_ground:
            self.y_velocity = PLAYER_JUMP_STRENGTH
            self.on_ground = False

    def update(self):
        # Apply gravity
        self.y_velocity += GRAVITY
        self.rect.y += self.y_velocity

        # Check for ground collision
        if self.rect.y >= PLAYER_GROUND_Y:
            self.rect.y = PLAYER_GROUND_Y
            self.y_velocity = 0
            self.on_ground = True

class Obstacle(pygame.sprite.Sprite):
    def __init__(self, x, y, width, height, color):
        super().__init__()
        self.image = pygame.Surface([width, height])
        self.image.fill(color)
        self.rect = self.image.get_rect()
        self.rect.x = x
        self.rect.y = y

    def update(self, game_speed):
        self.rect.x -= game_speed
        if self.rect.right < 0: # Remove obstacle when it goes off screen
            self.kill()

class GroundSpike(Obstacle):
    def __init__(self, x):
        width = random.randint(OBSTACLE_MIN_WIDTH, OBSTACLE_MAX_WIDTH)
        height = random.randint(OBSTACLE_MIN_HEIGHT, OBSTACLE_MAX_HEIGHT // 2) # Ground spikes are usually shorter
        y = SCREEN_HEIGHT - GROUND_HEIGHT - height
        super().__init__(x, y, width, height, RED) # Ground spike color

class FlyingEnemy(Obstacle):
    def __init__(self, x):
        width = FLYING_ENEMY_WIDTH
        height = FLYING_ENEMY_HEIGHT
        y = FLYING_ENEMY_Y - random.randint(-20, 20) # Slightly vary flying height
        super().__init__(x, y, width, height, GREEN) # Flying enemy color

# --- Set up the display ---
screen = pygame.display.set_mode((SCREEN_WIDTH, SCREEN_HEIGHT))
pygame.display.set_caption("Pixel Leaper")

# --- Scrolling Background/Ground Elements Surfaces ---
# Ground surface
ground_surf = pygame.Surface([SCREEN_WIDTH, GROUND_HEIGHT])
ground_surf.fill(BROWN)

# Mountain surface (simple triangular shape)
mountain_width_total = int(SCREEN_WIDTH * 1.5) # Mountains are wider for slower parallax
mountain_height = int(SCREEN_HEIGHT - GROUND_HEIGHT - BACKGROUND_MOUNTAIN_Y)
mountain_surf = pygame.Surface([mountain_width_total, mountain_height], pygame.SRCALPHA)
# Draw a simple mountain range profile
pygame.draw.polygon(mountain_surf, MOUNTAIN_COLOR, [
    (0, mountain_height),
    (mountain_width_total * 0.2, mountain_height * 0.1),
    (mountain_width_total * 0.4, mountain_height),
    (mountain_width_total * 0.6, mountain_height * 0.3),
    (mountain_width_total * 0.8, mountain_height * 0.05),
    (mountain_width_total, mountain_height)
])

# Cloud surface (simple blob shapes)
cloud_width_total = int(SCREEN_WIDTH * 2) # Clouds are even wider for slowest parallax
cloud_height = 60
cloud_surf = pygame.Surface([cloud_width_total, cloud_height], pygame.SRCALPHA)
# Draw simple cloud shapes
pygame.draw.ellipse(cloud_surf, CLOUD_COLOR, (0, 0, cloud_width_total * 0.4, cloud_height * 0.8))
pygame.draw.ellipse(cloud_surf, CLOUD_COLOR, (cloud_width_total * 0.2, cloud_height * 0.2, cloud_width_total * 0.5, cloud_height * 0.8))
pygame.draw.ellipse(cloud_surf, CLOUD_COLOR, (cloud_width_total * 0.5, 0, cloud_width_total * 0.3, cloud_height * 0.7))
pygame.draw.ellipse(cloud_surf, CLOUD_COLOR, (cloud_width_total * 0.7, cloud_height * 0.1, cloud_width_total * 0.2, cloud_height * 0.6))


# --- Game Variables ---
clock = pygame.time.Clock()
font = pygame.font.Font(None, 36) # Default font, size 36

# Sprite groups
all_sprites = pygame.sprite.Group()
obstacles = pygame.sprite.Group()

# Player instance (will be created in reset_game_state)
player = None

game_over = False
score = 0
game_speed = INITIAL_GAME_SPEED
last_obstacle_spawn_time = 0
game_start_time = 0

# Scrolling background elements' current X positions
# Using two instances for seamless scrolling
ground_x1 = 0
ground_x2 = SCREEN_WIDTH

mountain_x1 = 0
mountain_x2 = mountain_width_total

cloud_x1 = 0
cloud_x2 = cloud_width_total


# --- Game Reset Function ---
# This function will be called to initialize or restart the game state
def reset_game_state():
    global player, all_sprites, obstacles, game_over, score, game_speed, last_obstacle_spawn_time, game_start_time
    global ground_x1, ground_x2, mountain_x1, mountain_x2, cloud_x1, cloud_x2 # Declare as global to modify

    game_over = False
    score = 0
    game_speed = INITIAL_GAME_SPEED

    all_sprites.empty() # Clear all existing sprites
    obstacles.empty()

    player = Leapster() # Recreate player
    all_sprites.add(player)

    game_start_time = pygame.time.get_ticks() # Reset game start time
    last_obstacle_spawn_time = pygame.time.get_ticks() # Reset obstacle spawn timer

    # Reset scrolling background positions
    ground_x1 = 0
    ground_x2 = SCREEN_WIDTH
    mountain_x1 = 0
    mountain_x2 = mountain_width_total
    cloud_x1 = 0
    cloud_x2 = cloud_width_total

# Initial call to set up the game state when the script starts
reset_game_state()

# --- Game Loop ---
running = True
while running:
    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            running = False
        if event.type == pygame.KEYDOWN:
            if event.key == pygame.K_SPACE or event.key == pygame.K_UP:
                if not game_over:
                    player.jump()
                else: # Restart game if game over
                    reset_game_state()

    if not game_over:
        # --- Update ---
        all_sprites.update()

        # Update score (1 point per 100ms survived)
        score = (pygame.time.get_ticks() - game_start_time) // 100

        # Increase game speed (capped at MAX_GAME_SPEED)
        game_speed = min(INITIAL_GAME_SPEED + (score * GAME_SPEED_INCREMENT_PER_SCORE), MAX_GAME_SPEED)

        # Update scrolling background/ground positions
        ground_x1 -= game_speed
        ground_x2 -= game_speed
        if ground_x1 <= -SCREEN_WIDTH: ground_x1 = SCREEN_WIDTH + ground_x2 % SCREEN_WIDTH
        if ground_x2 <= -SCREEN_WIDTH: ground_x2 = SCREEN_WIDTH + ground_x1 % SCREEN_WIDTH

        mountain_x1 -= game_speed * PARALLAX_SPEED_FACTOR_MOUNTAINS
        mountain_x2 -= game_speed * PARALLAX_SPEED_FACTOR_MOUNTAINS
        if mountain_x1 <= -mountain_width_total: mountain_x1 = mountain_width_total + mountain_x2 % mountain_width_total
        if mountain_x2 <= -mountain_width_total: mountain_x2 = mountain_width_total + mountain_x1 % mountain_width_total

        cloud_x1 -= game_speed * PARALLAX_SPEED_FACTOR_CLOUDS
        cloud_x2 -= game_speed * PARALLAX_SPEED_FACTOR_CLOUDS
        if cloud_x1 <= -cloud_width_total: cloud_x1 = cloud_width_total + cloud_x2 % cloud_width_total
        if cloud_x2 <= -cloud_width_total: cloud_x2 = cloud_width_total + cloud_x1 % cloud_width_total


        # Obstacle generation
        current_time = pygame.time.get_ticks()

        # Adjust spawn intervals based on score
        effective_min_interval = max(MIN_SPAWN_INTERVAL_CAP, INITIAL_OBSTACLE_SPAWN_MIN_INTERVAL - score * SPAWN_INTERVAL_DECREMENT_PER_SCORE)
        effective_max_interval = max(MIN_SPAWN_INTERVAL_CAP + 200, INITIAL_OBSTACLE_SPAWN_MAX_INTERVAL - score * SPAWN_INTERVAL_DECREMENT_PER_SCORE)

        if current_time - last_obstacle_spawn_time > random.randint(effective_min_interval, effective_max_interval):
            # Ensure there's a minimum gap before spawning a new obstacle
            # This prevents obstacles from spawning on top of each other
            can_spawn = True
            if obstacles:
                # Check if the right-most obstacle is far enough to the left
                # `sprites()[-1]` is generally the last added, but not necessarily the right-most after some time.
                # A more robust check might iterate `obstacles` for the actual right-most.
                # However, for continuous spawn from the right, `[-1]` works for *last added*.
                if SCREEN_WIDTH - obstacles.sprites()[-1].rect.right < MIN_GAP_BETWEEN_OBSTACLES:
                    can_spawn = False

            if can_spawn:
                if random.random() < 0.6: # 60% chance for ground spike
                    new_obstacle = GroundSpike(SCREEN_WIDTH)
                else: # 40% chance for flying enemy
                    new_obstacle = FlyingEnemy(SCREEN_WIDTH)
                all_sprites.add(new_obstacle)
                obstacles.add(new_obstacle)
                last_obstacle_spawn_time = current_time

        # Collision detection
        if pygame.sprite.spritecollideany(player, obstacles):
            game_over = True

    # --- Draw ---
    screen.fill(LIGHT_BLUE) # Sky background

    # Draw scrolling background elements (from furthest to closest)
    screen.blit(cloud_surf, (cloud_x1, BACKGROUND_CLOUD_Y))
    screen.blit(cloud_surf, (cloud_x2, BACKGROUND_CLOUD_Y))
    screen.blit(mountain_surf, (mountain_x1, BACKGROUND_MOUNTAIN_Y))
    screen.blit(mountain_surf, (mountain_x2, BACKGROUND_MOUNTAIN_Y))

    # Draw scrolling ground
    screen.blit(ground_surf, (ground_x1, SCREEN_HEIGHT - GROUND_HEIGHT))
    screen.blit(ground_surf, (ground_x2, SCREEN_HEIGHT - GROUND_HEIGHT))

    all_sprites.draw(screen) # Draw player and obstacles

    # Display score
    score_text = font.render(f"Score: {score}", True, BLACK)
    screen.blit(score_text, (SCREEN_WIDTH - score_text.get_width() - 10, 10))

    if game_over:
        game_over_text = font.render("GAME OVER!", True, BLACK)
        restart_text = font.render("Press SPACE or UP to Restart", True, BLACK)
        screen.blit(game_over_text, (SCREEN_WIDTH // 2 - game_over_text.get_width() // 2, SCREEN_HEIGHT // 2 - 20))
        screen.blit(restart_text, (SCREEN_WIDTH // 2 - restart_text.get_width() // 2, SCREEN_HEIGHT // 2 + 20))

    # Flip the display
    pygame.display.flip()

    # Cap the frame rate
    clock.tick(FPS)

pygame.quit()
sys.exit()



Writing main.py


In [ ]:
!pip install pygbag pyngrok -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.1/176.1 kB 3.9 MB/s eta 0:00:00


In [ ]:
!ngrok authtoken '3Bmsi9N9i0XgQp34bh7E6hV4WyC_6hhhi5cdvLfptPRtUnB61'

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [ ]:
import subprocess
import time
from pyngrok import ngrok

# Step 1: Build the game
print("🔨 Building game...")
build_result = subprocess.run(
    ['pygbag', '--build', '--version', '0.9', '--PYBUILD', '3.12', '--cdn', 'https://pygame-web.github.io/archives/0.9/', 'main.py'],
    capture_output=True, text=True
)
print(build_result.stdout)

if build_result.returncode != 0:
    print("❌ Build failed:")
    print(build_result.stderr)
else:
    print("✅ Build successful!")

    # Step 2: Start HTTP server
    print("\n🚀 Starting server...")
    server = subprocess.Popen(
        ['python', '-m', 'http.server', '8000', '--directory', '/content/build/web'],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE
    )

    # Wait for server to start
    time.sleep(5)

    # Step 3: Create ngrok tunnel
    try:
        print("🌐 Creating public URL...")
        public_url = ngrok.connect(8000)

        print("\n" + "="*60)
        print("🎮 YOUR GAME IS READY!")
        print("="*60)
        print(f"\n🔗 Click here to play: {public_url}")
        print("\n📝 How to play:")
        print("   • Press SPACE or Click to jump")
        print("="*60)

    except Exception as e:
        print(f"\n❌ Error creating tunnel: {e}")
        print("\n💡 Make sure you ran Cell 3 with a valid ngrok token")

🔨 Building game...

Serving python files from [/content/build/web]

with no security/performance in mind, i'm just a test tool : don't rely on me


SUMMARY
________________________

# the app folder
app_folder=/content

# artefacts directory
build_dir=/content/build/web

# cache directory
cache=/content/build/web-cache

# the window title and icon name
app_name=content

# package name, better make it unique
package=web.pygame.content-1776312647

# icons:  96x96 for desktop, 16x16 for web
icon=favicon.png

# js/wasm provider
cdn=https://pygame-web.github.io/archives/0.9/

now packing application ....


Ignored dirs: []
Ignored files: []
Now in /
     /main.py
Now in /sample_data
     /sample_data/README.md
     /sample_data/anscombe.json
     /sample_data/california_housing_train.csv
     /sample_data/mnist_test.csv
     /sample_data/california_housing_test.csv
     /sample_data/mnist_train_small.csv
optimizing /content
	/content : main.py
	/content : sample_data/README.md
	/content : s